In [2]:
!pip install nltk scikit-learn requests joblib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import re
import requests
import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

import joblib


In [4]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rounak\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

#### Load the ISOT Dataset

In [6]:
fake = pd.read_csv("ISOT/Fake.csv")
true = pd.read_csv("ISOT/True.csv")

fake["label"] = 0   # Fake news
true["label"] = 1   # Real news


In [7]:
print(fake.duplicated().sum())
print(true.duplicated().sum())

3
206


In [8]:
fake = fake.drop_duplicates(subset=["text"])
fake = fake.reset_index(drop=True)

print("New shape:", fake.shape)

true = true.drop_duplicates(subset=["text"])
true = true.reset_index(drop=True)

print("New shape:", true.shape)

New shape: (17455, 5)
New shape: (21192, 5)


In [9]:
fake = fake[["text", "label"]]
true = true[["text", "label"]]

In [10]:
fake["text"] = fake["text"].apply(clean_text)
true["text"] = true["text"].apply(clean_text)


In [11]:
data = pd.concat([fake, true], ignore_index=True)
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

print("Total samples:", data.shape)

Total samples: (38647, 2)


In [12]:
X_ISOT = data["text"]
y_ISOT = data["label"]

#### Load the LIAR Dataset

In [13]:
liar = pd.read_csv("LIAR/train.tsv", sep="\t", header=None)

# Column 1 = label
# Column 2 = statement text
liar = liar[[1,2]]
liar.columns = ["label", "text"]

print(liar["label"].value_counts())

label
half-true      2114
false          1995
mostly-true    1962
true           1676
barely-true    1654
pants-fire      839
Name: count, dtype: int64


Labeling similarly so that we can concat 2 datasets 
###### 0 for fake & 1 for real

In [14]:
fake_labels= ["pants-fire", "false", "barely-true"]
real_labels = ["half-true", "mostly-true", "true"]

liar = liar[liar["label"].isin(fake_labels + real_labels)]

liar["label"] = liar["label"].apply(
    lambda x: 0 if x in fake_labels else 1
)

print(liar["label"].value_counts())

label
1    5752
0    4488
Name: count, dtype: int64


In [15]:
X_LIAR = liar["text"].apply(clean_text)
y_LIAR = liar["label"]

Combine the 2 datsets

In [16]:
X_combined = pd.concat([X_LIAR, X_ISOT], ignore_index=True)
y_combined = pd.concat([y_LIAR, y_ISOT], ignore_index=True)

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_combined,
    y_combined,
    test_size=0.2,
    stratify=y_combined,
    random_state=42
)

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [19]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

svm_model = LinearSVC()
svm_model.fit(X_train_vec, y_train)

svm_pred = svm_model.predict(X_test_vec)

print("SVM Accuracy:", accuracy_score(y_test, svm_pred))
print(classification_report(y_test, svm_pred))

SVM Accuracy: 0.9074452853344243
              precision    recall  f1-score   support

           0       0.91      0.89      0.90      4389
           1       0.91      0.93      0.92      5389

    accuracy                           0.91      9778
   macro avg       0.91      0.91      0.91      9778
weighted avg       0.91      0.91      0.91      9778



In [20]:
from sklearn.linear_model import PassiveAggressiveClassifier

pa_model = PassiveAggressiveClassifier(max_iter=1000)
pa_model.fit(X_train_vec, y_train)

pa_pred = pa_model.predict(X_test_vec)

print("Passive Aggressive Accuracy:", accuracy_score(y_test, pa_pred))
print(classification_report(y_test, pa_pred))

Passive Aggressive Accuracy: 0.89578645939865
              precision    recall  f1-score   support

           0       0.89      0.88      0.88      4389
           1       0.90      0.91      0.91      5389

    accuracy                           0.90      9778
   macro avg       0.90      0.89      0.89      9778
weighted avg       0.90      0.90      0.90      9778



In [21]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)

nb_pred = nb_model.predict(X_test_vec)

print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_pred))
print(classification_report(y_test, nb_pred))

Naive Bayes Accuracy: 0.8700143178564124
              precision    recall  f1-score   support

           0       0.86      0.85      0.85      4389
           1       0.88      0.89      0.88      5389

    accuracy                           0.87      9778
   macro avg       0.87      0.87      0.87      9778
weighted avg       0.87      0.87      0.87      9778



In [28]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=50,     # reduce trees
    max_depth=20,        # limit depth
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_vec, y_train)

rf_pred = rf_model.predict(X_test_vec)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))

Random Forest Accuracy: 0.8924115361014522
              precision    recall  f1-score   support

           0       0.97      0.79      0.87      4389
           1       0.85      0.98      0.91      5389

    accuracy                           0.89      9778
   macro avg       0.91      0.88      0.89      9778
weighted avg       0.90      0.89      0.89      9778



soft voting(majority)

In [29]:
from sklearn.ensemble import VotingClassifier

voting_clf = VotingClassifier(
    estimators=[
        ('svm', svm_model),
        ('pa', pa_model),
        ('nb', nb_model),
        ('rf', rf_model)
    ],
    voting='hard'
)

voting_clf.fit(X_train_vec, y_train)


vote_pred = voting_clf.predict(X_test_vec)

print("Voting Classifier Accuracy:", accuracy_score(y_test, vote_pred))
print(classification_report(y_test, vote_pred))

Voting Classifier Accuracy: 0.9055021476784618
              precision    recall  f1-score   support

           0       0.90      0.89      0.89      4389
           1       0.91      0.92      0.91      5389

    accuracy                           0.91      9778
   macro avg       0.91      0.90      0.90      9778
weighted avg       0.91      0.91      0.91      9778



save these models

In [30]:
import joblib

joblib.dump(svm_model, "svm_model.pkl")
joblib.dump(pa_model, "pa_model.pkl")
joblib.dump(nb_model, "nb_model.pkl")
joblib.dump(rf_model, "rf_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

joblib.dump(voting_clf, "voting_model.pkl")

['voting_model.pkl']